# DurakZero Training Notebook

This notebook provides an end-to-end workflow to train and evaluate the DurakZero agent using the Deep Monte-Carlo self-play pipeline.

## 1. Environment setup

Run the following cell to verify the environment, add the project to the Python path, and inspect the available hardware.

In [ ]:
import os
import sys
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for idx in range(torch.cuda.device_count()):
        print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)}")


If you are running in a fresh environment (for example on a new cloud instance), install the project requirements.

```python
# import os, subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", os.path.join(PROJECT_ROOT, "requirements.txt")])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
```

Uncomment the commands above only if the dependencies are not yet installed.

## 2. Configure a training run

Use the helper below to create a configuration object (`flags`) that mirrors the command-line arguments exposed by `train.py`. Override any defaults that you want to customize for your hardware. For large-scale runs (e.g., on an H100), increase `num_actor_devices`, `num_actors`, and `total_frames`.

In [ ]:
from douzero.dmc import parser

def make_flags(overrides=None):
    overrides = overrides or {}
    flags = parser.parse_args(args=[])

    defaults = dict(
        actor_device_cpu=not torch.cuda.is_available(),
        gpu_devices="0",
        num_actor_devices=1,
        num_actors=8,
        training_device="0" if torch.cuda.is_available() else "cpu",
        total_frames=200_000_000,
        batch_size=32,
        unroll_length=80,
        learning_rate=1e-4,
        save_interval=30,
        xpid="durakzero_notebook",
        savedir=os.path.join(PROJECT_ROOT, "durakzero_checkpoints"),
    )

    defaults.update(overrides)

    for key, value in defaults.items():
        setattr(flags, key, value)

    os.makedirs(flags.savedir, exist_ok=True)

    print("Training configuration:")
    for key in sorted(defaults.keys()):
        print(f"  {key}: {getattr(flags, key)}")
    return flags

# Example overrides for CPU-only training. Adjust as needed.
flags = make_flags({
    "actor_device_cpu": True,
    "training_device": "cpu",
    "gpu_devices": "",
    "total_frames": 2_000_000,
    "num_actors": 4,
})


> **Tip:** When training on a powerful GPU such as an H100, set `actor_device_cpu=False`, choose the GPU IDs in `gpu_devices` (e.g., `'0,1,2,3'`), and distribute actors by increasing `num_actor_devices` and `num_actors`. Keep `training_device` equal to the GPU that should host the learner.

## 3. Launch training

Execute the next cell to start self-play training. The loop runs until `total_frames` is reached; you can interrupt the cell at any time, and DurakZero will save the most recent checkpoint.

In [ ]:
from douzero.dmc import train

if flags.gpu_devices:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(flags.gpu_devices)

try:
    train(flags)
except KeyboardInterrupt:
    print("Training interrupted manually. The latest checkpoint (if any) remains on disk.")


## 4. Evaluate a checkpoint

Once a checkpoint (`model.tar`) has been produced in the save directory, evaluate it via self-play to estimate the win rate for each Durak seat.

In [ ]:
import json
from pathlib import Path

from douzero.evaluation.simulation import evaluate

checkpoint_path = Path(flags.savedir) / flags.xpid / "model.tar"
print(f"Checkpoint path: {checkpoint_path}")

if checkpoint_path.exists():
    results = evaluate(str(checkpoint_path), num_games=1000)
    print(json.dumps(results, indent=2))
else:
    print("No checkpoint found yet. Run training long enough to produce one (see flags.save_interval).")


## 5. Continue training later

To resume from an existing run, keep the same `xpid` and enable `load_model=True` in the overrides when creating `flags`:

```python
flags = make_flags({
    "xpid": "durakzero_notebook",
    "load_model": True,
})
```

DurakZero will load the checkpoint in `flags.savedir/flags.xpid/model.tar` and continue training from the saved state.

## 6. Notes

- The Deep Monte-Carlo algorithm relies on a large number of self-play trajectories. Expect to train for billions of frames to reach superhuman performance; scale `total_frames`, `num_actor_devices`, and `num_actors` accordingly.
- Checkpoints and training logs are written under `flags.savedir/flags.xpid/`. Use the generated `log.csv` to visualize learning curves with your preferred plotting tool.
- Adjust `exp_epsilon` in `make_flags` to control exploration when you are confident in the policy.